# Extract Measures — Stimulus-Locked (Flow & Load papers)

Reads from pkl files produced by `grand_average.ipynb` and exports:
- one CSV per measure type (raw / diff)
- format: long, all channels, all windows — filter in R

Windows are loaded from `config_flow.py` / `config_load.py` (`WINDOWS` dict).

**Usage**: set `PAPER`, `PARADIGM` and `TRIALS` in Cell 1, then run all.

`TRIALS` selects the grand-average subfolder (`raw/{TRIALS}/`, `diff/{TRIALS}/`) and is
written into the output CSV names.

Last edit: 31.10.2026 @anguyen

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys

import pandas as pd
from tqdm import tqdm

def find_repo_root(marker='core') -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root() / 'core'))
from config_flow import WINDOWS as WINDOWS_FLOW
from config_load import WINDOWS as WINDOWS_LOAD
import config_flow, config_load
from participants import get_participants
from utils import load_pkl

# ==============================================================================
# CONFIGURATION — edit here
# ==============================================================================

PAPER:    str = 'flow'    # 'flow' | 'load'
PARADIGM: str = 'active'  # 'active' | 'passive'

TRIALS = 'correct'   # 'correct' (primary) | 'alltrials'

# ==============================================================================
# FIXED — do not change
# ==============================================================================

LOCK  = 'stim'
STUDY = 'Active' if PARADIGM == 'active' else 'Passive'

if sys.platform == 'win32':
    root = Path(f'L:/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2')
else:
    root = Path(f'/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2')

assert TRIALS in ('correct', 'alltrials'), f'Unknown TRIALS: {TRIALS}'

input_path_raw  = root / 'outputs' / 'grand_averages' / PAPER / 'raw'  / TRIALS
input_path_diff = root / 'outputs' / 'grand_averages' / PAPER / 'diff' / TRIALS


def latest_date(folder: Path, pattern: str) -> str:
    """Most recent date prefix among pkl files matching pattern in folder."""
    files = sorted(folder.glob(pattern))
    assert files, f'No pkl matching {pattern} in {folder}'
    dates = sorted({f.name.split('_')[0] for f in files})
    print(f'Available dates in {folder.relative_to(root)}: {dates}')
    return dates[-1]


DATE_SOURCE = latest_date(input_path_raw, f'*_{PARADIGM}_*p_cond-*_stim_raw.pkl')
print(f'Paper: {PAPER.upper()} | Paradigm: {PARADIGM.upper()} | Lock: {LOCK.upper()} | '
      f'Trials: {TRIALS} | Source date: {DATE_SOURCE}')

In [ ]:
# ==============================================================================
# Derived labels and paths (auto — do not edit)
# ==============================================================================

_cfg = config_flow if PAPER == 'flow' else config_load

CONDITIONS    = _cfg.CONDITIONS
DEVIANT_LABEL = 'target' if PARADIGM == 'active' else 'deviant'
DATE_STR      = datetime.today().strftime('%Y%m%d')

n = len(get_participants(PAPER, PARADIGM))

output_path     = root / 'outputs' / 'measures' / PAPER
output_path.mkdir(parents=True, exist_ok=True)

stim_events = ['standard', DEVIANT_LABEL, 'novel']
diff_events = [f'{DEVIANT_LABEL}-standard', 'novel-standard']
if PAPER == 'flow':
    diff_events.append(f'{DEVIANT_LABEL}-novel')

_WINDOWS = WINDOWS_FLOW if PAPER == 'flow' else WINDOWS_LOAD
WINDOWS_MS_RAW  = _WINDOWS.get((PARADIGM, LOCK, 'raw'),  [])
WINDOWS_MS_DIFF = _WINDOWS.get((PARADIGM, LOCK, 'diff'), [])
WINDOWS_S_RAW   = [(t1 / 1000, t2 / 1000) for t1, t2 in WINDOWS_MS_RAW]
WINDOWS_S_DIFF  = [(t1 / 1000, t2 / 1000) for t1, t2 in WINDOWS_MS_DIFF]

print(f'Input raw   : {input_path_raw}')
print(f'Input diff  : {input_path_diff}')
print(f'Output      : {output_path}')
print(f'Windows raw : {WINDOWS_MS_RAW}')
print(f'Windows diff: {WINDOWS_MS_DIFF}')

In [ ]:
# ==============================================================================
# Load pkl files
# ==============================================================================

def _build_fname(cond_key: str, measure: str) -> str:
    base = f'{DATE_SOURCE}_{PARADIGM}_{n}p_cond-{cond_key}'
    return f'{base}_stim_diff.pkl' if measure == 'diff' else f'{base}_stim_raw.pkl'


evokeds_raw:  dict[str, dict] = {}
evokeds_diff: dict[str, dict] = {}

for plot_label, cond_key in CONDITIONS.items():
    evokeds_raw[plot_label]  = load_pkl(input_path_raw  / _build_fname(cond_key, 'raw'))
    evokeds_diff[plot_label] = load_pkl(input_path_diff / _build_fname(cond_key, 'diff'))
    print(f'Loaded [{plot_label}]')
    print(f'  raw keys  : {list(evokeds_raw[plot_label].keys())}')
    print(f'  diff keys : {list(evokeds_diff[plot_label].keys())}')

In [ ]:
# ==============================================================================
# Extraction function
# ==============================================================================

def extract_mean_amplitude(
    evokeds_dict: dict[str, dict],
    event_list:   list[str],
    windows_s:    list[tuple],
    stim_col:     str = 'stim',
) -> pd.DataFrame:
    """Extract mean amplitude per participant, condition, event, channel, window.

    Parameters
    ----------
    evokeds_dict : {plot_label: {cond_event_key: [Evoked, ...]}}
    event_list   : event types to extract, e.g. ['standard', 'target', 'novel']
    windows_s    : list of (tmin, tmax) in seconds
    stim_col     : column name — 'stim' for raw, 'contrast' for diff

    Returns
    -------
    pd.DataFrame with columns:
        participant | condition | {stim_col} | ch_name | window | mean_amp_uv
    """
    rows = []

    for plot_label, cond_key in tqdm(CONDITIONS.items(), desc='Conditions', unit='cond'):
        data = evokeds_dict[plot_label]

        for event in event_list:
            key = f'{cond_key}_{event}'
            if key not in data or not data[key]:
                print(f"Warning: key '{key}' not found or empty — skipping.")
                continue

            for i, evoked in enumerate(data[key]):
                participant = evoked.comment if evoked.comment else f'sub-{i:03d}'

                for tmin, tmax in windows_s:
                    window = f'{int(tmin * 1000)}-{int(tmax * 1000)}ms'
                    try:
                        evoked_cropped = (
                            evoked.copy().pick('eeg', exclude='bads')
                            .crop(tmin=tmin, tmax=tmax)
                        )
                    except Exception as e:
                        print(f'Warning: crop failed for {participant} {event} {window} -> {e}')
                        continue

                    mean_amp = evoked_cropped.data.mean(axis=1) * 1e6

                    for ch_idx, ch_name in enumerate(evoked_cropped.ch_names):
                        rows.append({
                            'participant': participant,
                            'condition':   plot_label,
                            stim_col:      event,
                            'ch_name':     ch_name,
                            'window':      window,
                            'mean_amp_uv': mean_amp[ch_idx],
                        })

    return pd.DataFrame(rows)

print('extract_mean_amplitude() ready.')

## Extract raw

In [ ]:
df_raw = extract_mean_amplitude(
    evokeds_dict = evokeds_raw,
    event_list   = stim_events,
    windows_s    = WINDOWS_S_RAW,
    stim_col     = 'stim',
)

n_participants = df_raw['participant'].nunique()
print(f'\nRaw: {len(df_raw):,} rows | {n_participants} participants')
print(df_raw.head())

fname_raw = output_path / f'{DATE_STR}_{PARADIGM}_{n_participants}p_{PAPER}_{LOCK}_raw_{TRIALS}_allch_allwin_meanamp.csv'
df_raw.to_csv(fname_raw, index=False)
print(f'\nSaved -> {fname_raw}')

## Extract diff

In [ ]:
df_diff = extract_mean_amplitude(
    evokeds_dict = evokeds_diff,
    event_list   = diff_events,
    windows_s    = WINDOWS_S_DIFF,
    stim_col     = 'contrast',
)

n_participants = df_diff['participant'].nunique()
print(f'\nDiff: {len(df_diff):,} rows | {n_participants} participants')
print(df_diff.head())

fname_diff = output_path / f'{DATE_STR}_{PARADIGM}_{n_participants}p_{PAPER}_{LOCK}_diff_{TRIALS}_allch_allwin_meanamp.csv'
df_diff.to_csv(fname_diff, index=False)
print(f'\nSaved -> {fname_diff}')

## Sanity checks

1. Structure: participants, conditions, events, channels, windows, and no missing values.
2. Outliers: |z| > 3 or |amplitude| > 40 µV at the electrodes of interest.

The comparison with v1 now lives in `compare_step8_versions.ipynb` (evoked level, true v1 = `step70dd`).

In [ ]:
for label, df in [('RAW', df_raw), ('DIFF', df_diff)]:
    print(f'\n=== {label} ===')
    print(f'  Rows         : {len(df):,}')
    print(f'  Participants : {df["participant"].nunique()}')
    print(f'  Conditions   : {sorted(df["condition"].unique())}')
    stim_col = 'stim' if 'stim' in df.columns else 'contrast'
    print(f'  Events       : {sorted(df[stim_col].unique())}')
    print(f'  Channels     : {df["ch_name"].nunique()}')
    print(f'  Windows      : {sorted(df["window"].unique())}')
    print(f'  Any NaN      : {df["mean_amp_uv"].isna().any()}')

In [ ]:
from scipy import stats

ZSCORE_THRESHOLD = 3.0
AMP_THRESHOLD_UV = 40.0

ELECTRODES_OF_INTEREST = list(set([
    'F1', 'Fz', 'F2', 'FC1', 'FCz', 'FC2',
    'C1', 'Cz', 'C2', 'CP1', 'CPz', 'CP2',
    'P1', 'Pz', 'P2', 'POz',
    'F3', 'FC3', 'F4', 'FC4',
]))

for label, df in [('RAW', df_raw), ('DIFF', df_diff)]:
    stim_col = 'stim' if 'stim' in df.columns else 'contrast'

    df_interest = df[df['ch_name'].isin(ELECTRODES_OF_INTEREST)].copy()
    df_interest['zscore'] = df_interest.groupby(
        ['condition', stim_col, 'window', 'ch_name']
    )['mean_amp_uv'].transform(lambda x: stats.zscore(x, nan_policy='omit'))

    outliers_z   = df_interest[df_interest['zscore'].abs() > ZSCORE_THRESHOLD]
    outliers_amp = df_interest[df_interest['mean_amp_uv'].abs() > AMP_THRESHOLD_UV]
    outliers     = pd.concat([outliers_z, outliers_amp]).drop_duplicates()

    print(f'\n=== {label} — Outliers ===')
    print(f'  Z-score  (|z|>{ZSCORE_THRESHOLD})   : {len(outliers_z)} rows')
    print(f'  Amplitude (|uV|>{AMP_THRESHOLD_UV}) : {len(outliers_amp)} rows')
    print(f'  Total unique flagged                 : {len(outliers)} rows')
    if len(outliers) > 0:
        print(f'  Participants affected : {sorted(outliers["participant"].unique())}')
        print('\n  Top 10 worst:')
        print(
            outliers[['participant', 'condition', stim_col, 'ch_name', 'window', 'mean_amp_uv', 'zscore']]
            .sort_values('zscore', key=abs, ascending=False)
            .head(10)
            .to_string(index=False)
        )